# Diabetes Data Cleaning & Feature Engineering

A compact data-preparation case study using a diabetes dataset with biologically implausible zero values encoded in several clinical measurements.

The workflow identifies these values, converts them to missing data, applies mean imputation, creates a pregnancy-history indicator, and summarizes the observed diabetes outcome rate for participants with one or more recorded pregnancies.

**Tools:** Python · Pandas · NumPy


## 1. Load and inspect the data

The dataset is expected at `data/diabetes.csv`.


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/diabetes.csv")
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


## 2. Identify zero-coded missing values

Several clinical measurements contain zeros that are not plausible measurements. These values are treated as missing for the cleaning workflow.


In [ ]:
clinical_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]

zero_counts = (df[clinical_columns] == 0).sum()
zero_counts


,0
Pregnancies,111
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11
DiabetesPedigreeFunction,0
Age,0
Outcome,500


## 3. Convert invalid zeros to missing values and impute

Zeros in the selected clinical columns are converted to `NaN`, then missing values are filled with the mean of each numeric column.


In [ ]:
df[clinical_columns] = df[clinical_columns].replace(0, np.nan)

missing_before_imputation = df.isna().sum()
missing_before_imputation


,0
Pregnancies,0
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [ ]:
df = df.fillna(df.mean(numeric_only=True))

missing_after_imputation = df.isna().sum()
missing_after_imputation


,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


## 4. Create a pregnancy-history feature

A binary feature separates participants with at least one recorded pregnancy from those with zero recorded pregnancies.


In [ ]:
df["has_pregnancy_history"] = (df["Pregnancies"] > 0).astype(int)

df[["Pregnancies", "has_pregnancy_history", "Outcome"]].head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Pregnant_bool
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0,1
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0,1
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1,0


## 5. Outcome rate among participants with pregnancy history

This is a descriptive proportion in the dataset, not a causal estimate.


In [ ]:
outcome_rate = df.loc[
    df["has_pregnancy_history"].eq(1), "Outcome"
].mean()

print(f"Observed diabetes outcome rate: {outcome_rate:.1%}")


The proportion of individuals who develop diabetes following pregnancy is: 0.350076103500761


## 6. Key takeaways

- Missing-data problems can be hidden behind valid numeric types when sentinel values such as zero are used.
- Domain-aware cleaning should happen before statistical summaries or modeling.
- Mean imputation removes missing values but can reduce variance, so more sophisticated imputation may be preferable for predictive work.
- Among participants with at least one recorded pregnancy, the observed positive diabetes outcome rate in this dataset is about 35.0%.
